# FloraNER — WMC (Said & MS)
Run on a Colab T4 GPU. This notebook runs only the duo's baseline: seeds 1, 2, 3; 60 epochs maximum; patience 10; Adam lr=0.001; BiWMC H=128; dropout 0.3; batch 32. Same data pipeline and evaluation as Vanilla. Best validation epoch is evaluated on test. No collective comparison, report, or slides are generated.


In [ ]:
import os, sys, subprocess
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, then run again.'
print(torch.__version__, torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')
os.environ['FLORANER_RESULTS'] = '/content/drive/MyDrive/floraner_wmc_results_e60'
os.makedirs(os.environ['FLORANER_RESULTS'], exist_ok=True)
os.makedirs('/content/floraner/src', exist_ok=True)
os.makedirs('/content/floraner/tests', exist_ok=True)
os.chdir('/content/floraner')
def run(*args):
    command = [sys.executable, *args]
    print('Running:', ' '.join(command), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait() != 0:
        raise subprocess.CalledProcessError(process.returncode, command)
run('-m', 'pip', 'install', '-q', 'pytest', 'numpy')


In [ ]:
%%writefile src/data.py
"""FloraNER data pipeline: character spans -> BIO tokens -> padded PyTorch batches.

The raw FloraNER CSVs store entities as character offsets ``(start, end, label)``.
This module tokenizes each text, projects the spans onto tokens with the BIO
scheme, carves a document-level validation split out of the official train set
and exposes a ``Dataset`` / ``collate_fn`` pair shared by every model of the group.
"""
from __future__ import annotations

import ast
import csv
import json
import random
import re
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path

import torch
from torch.utils.data import Dataset

ROOT = Path(__file__).resolve().parent.parent
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"

# task name -> (file stem, annotation column)
TASKS = {
    "species": ("species", "Annotations"),
    "coarse": ("coarse_grained", "filtered_annotations"),
    "fine": ("fine_grained", "filtered_annotations"),
}

PAD, UNK = "<PAD>", "<UNK>"
PAD_LABEL_ID = -100  # ignored by nn.CrossEntropyLoss
VAL_RATIO = 0.10
SPLIT_SEED = 42

# Words and single punctuation marks. Splitting hyphens/apostrophes keeps 100% of
# the coarse/fine entity boundaries aligned with token boundaries.
TOKEN_RE = re.compile(r"\w+|[^\w\s]")

csv.field_size_limit(10**8)


# --------------------------------------------------------------------------- #
# Raw loading and BIO conversion
# --------------------------------------------------------------------------- #
def load_raw(task: str, split: str) -> list[tuple[str, list[tuple[int, int, str]]]]:
    """Return ``[(text, [(start, end, label), ...]), ...]`` for an official split."""
    stem, column = TASKS[task]
    folder = "Train" if split == "train" else "Test"
    path = RAW_DIR / folder / f"{split}_{stem}.csv"
    docs = []
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            cell = row[column].strip()
            spans = ast.literal_eval(cell) if cell else []
            docs.append((row["Text"], [tuple(s) for s in spans]))
    return docs


def tokenize(text: str) -> list[tuple[str, int, int]]:
    return [(m.group(), m.start(), m.end()) for m in TOKEN_RE.finditer(text)]


@dataclass
class ConversionStats:
    entities: int = 0
    snapped: int = 0  # span boundary fell inside a token -> extended to the whole token
    dropped_empty: int = 0  # span covers no token
    dropped_overlap: int = 0  # span overlaps an entity already placed


def spans_to_bio(text: str, spans, stats: ConversionStats | None = None):
    """Project character spans onto tokens. A token belongs to an entity if it overlaps it."""
    toks = tokenize(text)
    tags = ["O"] * len(toks)
    stats = stats if stats is not None else ConversionStats()
    for start, end, label in sorted(set(spans), key=lambda s: (s[0], -s[1])):
        stats.entities += 1
        idx = [i for i, (_, s, e) in enumerate(toks) if s < end and e > start]
        if not idx:
            stats.dropped_empty += 1
            continue
        if any(tags[i] != "O" for i in idx):
            stats.dropped_overlap += 1
            continue
        if toks[idx[0]][1] != start or toks[idx[-1]][2] != end:
            stats.snapped += 1
        tags[idx[0]] = f"B-{label}"
        for i in idx[1:]:
            tags[i] = f"I-{label}"
    return [t for t, _, _ in toks], tags, [(s, e) for _, s, e in toks]


def segment(tokens, tags, offsets, text, max_len: int):
    """Split a long document into chunks of at most ``max_len`` tokens.

    Cuts happen only between two tokens where no entity is interrupted, preferably
    after a clause boundary ('.', ';' or a run of 2+ spaces in the source text).
    Entity-level metrics are therefore identical on segments and on full documents.
    """
    if len(tokens) <= max_len:
        return [(tokens, tags)]

    def can_cut(i):  # cut between token i-1 and token i
        return not tags[i].startswith("I-")

    def is_boundary(i):
        gap = text[offsets[i - 1][1]:offsets[i][0]]
        return tokens[i - 1] in {".", ";"} or "  " in gap or "\n" in gap

    chunks, start = [], 0
    while len(tokens) - start > max_len:
        limit = start + max_len
        cut = next((i for i in range(limit, start, -1) if can_cut(i) and is_boundary(i)), None)
        if cut is None:
            cut = next(i for i in range(limit, start, -1) if can_cut(i))
        chunks.append((tokens[start:cut], tags[start:cut]))
        start = cut
    chunks.append((tokens[start:], tags[start:]))
    return chunks


def build_processed(task: str, max_len: int = 128) -> dict:
    """Write ``data/processed/<task>/{train,val,test}.jsonl`` and return conversion stats."""
    out_dir = PROCESSED_DIR / task
    out_dir.mkdir(parents=True, exist_ok=True)

    train_docs = load_raw(task, "train")
    test_docs = load_raw(task, "test")
    order = list(range(len(train_docs)))
    random.Random(SPLIT_SEED).shuffle(order)
    n_val = round(VAL_RATIO * len(order))
    val_ids, train_ids = set(order[:n_val]), set(order[n_val:])

    splits = {
        "train": [(i, train_docs[i]) for i in sorted(train_ids)],
        "val": [(i, train_docs[i]) for i in sorted(val_ids)],
        "test": list(enumerate(test_docs)),
    }
    report = {}
    for name, docs in splits.items():
        stats = ConversionStats()
        n_seg = 0
        with open(out_dir / f"{name}.jsonl", "w", encoding="utf-8") as f:
            for doc_id, (text, spans) in docs:
                tokens, tags, offsets = spans_to_bio(text, spans, stats)
                if not tokens:
                    continue
                for k, (seg_toks, seg_tags) in enumerate(segment(tokens, tags, offsets, text, max_len)):
                    f.write(json.dumps({"doc_id": doc_id, "seg": k, "tokens": seg_toks, "tags": seg_tags},
                                       ensure_ascii=False) + "\n")
                    n_seg += 1
        report[name] = {"documents": len(docs), "segments": n_seg, **stats.__dict__}
    with open(out_dir / "conversion_report.json", "w") as f:
        json.dump(report, f, indent=2)
    return report


def read_jsonl(task: str, split: str) -> list[dict]:
    with open(PROCESSED_DIR / task / f"{split}.jsonl", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


# --------------------------------------------------------------------------- #
# Vocabularies, casing feature, Dataset
# --------------------------------------------------------------------------- #
CASES = ["<pad>", "lower", "title", "upper", "mixed", "numeric", "has_digit", "punct"]


def casing(token: str) -> int:
    """Small orthographic feature: lowercased embeddings lose capitalisation (genus names)."""
    if token.isdigit():
        c = "numeric"
    elif any(ch.isdigit() for ch in token):
        c = "has_digit"
    elif not any(ch.isalnum() for ch in token):
        c = "punct"
    elif token.islower():
        c = "lower"
    elif token.isupper():
        c = "upper"
    elif token[0].isupper() and token[1:].islower():
        c = "title"
    else:
        c = "mixed"
    return CASES.index(c)


@dataclass
class Vocab:
    itos: list[str] = field(default_factory=list)

    def __post_init__(self):
        self.stoi = {s: i for i, s in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def __getitem__(self, token: str) -> int:
        return self.stoi.get(token, self.stoi.get(UNK, 0))

    @classmethod
    def build(cls, sequences, min_freq: int = 1, specials=(PAD, UNK)):
        counts = Counter(tok for seq in sequences for tok in seq)
        return cls(list(specials) + sorted(t for t, c in counts.items() if c >= min_freq and t not in specials))


def normalize(token: str, lowercase: bool) -> str:
    token = token.lower() if lowercase else token
    return re.sub(r"\d", "0", token)  # all digits share one shape: "12 mm" ~ "35 mm"


class NERDataset(Dataset):
    def __init__(self, records, word_vocab: Vocab, tag_vocab: Vocab, lowercase: bool = True):
        self.records = records
        self.word_vocab, self.tag_vocab, self.lowercase = word_vocab, tag_vocab, lowercase

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        r = self.records[i]
        words = torch.tensor([self.word_vocab[normalize(t, self.lowercase)] for t in r["tokens"]])
        cases = torch.tensor([casing(t) for t in r["tokens"]])
        tags = torch.tensor([self.tag_vocab.stoi[t] for t in r["tags"]])
        return words, cases, tags


def collate_fn(batch):
    """Right-pad to the longest sequence of the batch; labels are padded with -100."""
    words, cases, tags = zip(*batch)
    lengths = torch.tensor([len(w) for w in words])
    T = int(lengths.max())
    W = torch.zeros(len(batch), T, dtype=torch.long)
    C = torch.zeros(len(batch), T, dtype=torch.long)
    Y = torch.full((len(batch), T), PAD_LABEL_ID, dtype=torch.long)
    for i, (w, c, y) in enumerate(zip(words, cases, tags)):
        W[i, : len(w)], C[i, : len(c)], Y[i, : len(y)] = w, c, y
    mask = torch.arange(T)[None, :] < lengths[:, None]
    return W, C, Y, mask


def load_task(task: str, lowercase: bool = True, min_freq: int = 1):
    """Return train/val/test datasets plus vocabularies (built on the train split only)."""
    recs = {s: read_jsonl(task, s) for s in ("train", "val", "test")}
    word_vocab = Vocab.build([[normalize(t, lowercase) for t in r["tokens"]] for r in recs["train"]], min_freq)
    labels = sorted({t for r in recs["train"] for t in r["tags"]} - {"O"}, key=lambda t: (t[2:], t[0]))
    tag_vocab = Vocab(["O"] + labels)
    data = {s: NERDataset(r, word_vocab, tag_vocab, lowercase) for s, r in recs.items()}
    return data, word_vocab, tag_vocab


if __name__ == "__main__":
    import argparse

    p = argparse.ArgumentParser(description="Convert FloraNER CSVs into BIO jsonl splits.")
    p.add_argument("--tasks", nargs="+", default=list(TASKS))
    p.add_argument("--max-len", type=int, default=128)
    args = p.parse_args()
    for t in args.tasks:
        print(t, json.dumps(build_processed(t, args.max_len), indent=1))


In [ ]:
%%writefile src/download_data.py
"""Download FloraNER (Zenodo record 12765869, CC-BY-4.0) and build the BIO splits.

Usage: python src/download_data.py
"""
import urllib.request
import zipfile

from data import RAW_DIR, TASKS, build_processed

RECORD = "https://zenodo.org/api/records/12765869/files/{}/content"
FILES = ["Train.zip", "Test.zip"]


def main():
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    for name in FILES:
        target = RAW_DIR / name
        if not target.exists():
            print(f"downloading {name} ...")
            urllib.request.urlretrieve(RECORD.format(name), target)
        with zipfile.ZipFile(target) as z:
            z.extractall(RAW_DIR)
    for task in TASKS:
        report = build_processed(task)
        print(task, {split: (r["segments"], r["entities"]) for split, r in report.items()})


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/metrics.py
"""Entity-level NER metrics (conlleval / seqeval default semantics), no external dependency.

An entity is a maximal chunk (type, start, end). A predicted entity is correct only if its type and
both boundaries match a gold entity exactly. As in conlleval, an I-X that does not continue an X chunk
starts a new chunk. tests/test_model.py checks that the numbers equal seqeval's.
"""
from __future__ import annotations

from collections import Counter


def get_entities(tags: list[str]) -> list[tuple[str, int, int]]:
    """BIO tags -> [(type, start, end_inclusive), ...]."""
    entities, start, etype = [], None, None
    for i, tag in enumerate(list(tags) + ["O"]):
        prefix, t = ("O", None) if tag == "O" else (tag[0], tag[2:])
        if start is not None and (prefix in ("B", "O") or t != etype):
            entities.append((etype, start, i - 1))
            start = None
        if prefix in ("B", "I") and start is None:
            start, etype = i, t
    return entities


def _prf(tp: int, n_pred: int, n_true: int) -> tuple[float, float, float]:
    p = tp / n_pred if n_pred else 0.0
    r = tp / n_true if n_true else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f


def entity_scores(y_true: list[list[str]], y_pred: list[list[str]]) -> dict:
    """Micro/macro precision, recall, F1 and a per-type report over a corpus of sentences."""
    true = {(k, *e) for k, seq in enumerate(y_true) for e in get_entities(seq)}
    pred = {(k, *e) for k, seq in enumerate(y_pred) for e in get_entities(seq)}
    tp_by, true_by, pred_by = Counter(e[1] for e in true & pred), Counter(e[1] for e in true), \
        Counter(e[1] for e in pred)
    per_type = {}
    for t in sorted(set(true_by) | set(pred_by)):
        p, r, f = _prf(tp_by[t], pred_by[t], true_by[t])
        per_type[t] = {"precision": p, "recall": r, "f1-score": f, "support": true_by[t]}
    p, r, f = _prf(len(true & pred), len(pred), len(true))
    macro = sum(v["f1-score"] for v in per_type.values()) / len(per_type) if per_type else 0.0
    return {"precision": p, "recall": r, "f1": f, "macro_f1": macro, "per_type": per_type}


In [ ]:
%%writefile src/cells.py
"""Recurrent cells implemented from scratch with nn.Linear (nn.LSTM / nn.LSTMCell are not used).

Every cell follows the same small interface so that the sequence layer, the tagger and
the training script are shared by all the variants studied by the group:

    xp    = cell.input_projection(x)        # (B, T, D) -> (B, T, G): input terms for all t at once
    state = cell.init_state(B, device)      # tuple of tensors
    h, state = cell(xp[:, t], state)        # one recurrent step
    cell.output_size                        # size of h (what the next layer / classifier sees)

To add a variant (CIFG, LSTMP, WMC), subclass ``RecurrentCell`` and register it in ``CELLS``.
"""
from __future__ import annotations

import math

import torch
from torch import nn


class RecurrentCell(nn.Module):
    """Base class: one recurrent step, plus a hoisted input projection."""

    output_size: int

    def input_projection(self, x: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError

    def init_state(self, batch_size: int, device) -> tuple[torch.Tensor, ...]:
        raise NotImplementedError

    def reset_parameters(self, scheme: str = "default", forget_bias: float = 0.0) -> None:
        raise NotImplementedError

    def recurrent_parameter_count(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


class VanillaLSTMCell(RecurrentCell):
    r"""Vanilla LSTM, equations (1)-(6) of the guidelines.

        i_t = σ(W_hi h_{t-1} + W_xi x_t + b_i)
        f_t = σ(W_hf h_{t-1} + W_xf x_t + b_f)
        o_t = σ(W_ho h_{t-1} + W_xo x_t + b_o)
        g_t = tanh(W_hg h_{t-1} + W_xg x_t + b_g)
        c_t = f_t ⊙ c_{t-1} + i_t ⊙ g_t
        h_t = o_t ⊙ tanh(c_t)

    The four W_x* (resp. W_h*) matrices are stacked in one nn.Linear of output size 4H,
    in the order (i, f, g, o). This is the same maths as eight separate matrices, but a
    single matrix product per step. Biases live in ``x2h`` only (one bias per gate).
    """

    GATES = ("i", "f", "g", "o")

    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.input_size, self.hidden_size, self.output_size = input_size, hidden_size, hidden_size
        self.x2h = nn.Linear(input_size, 4 * hidden_size)               # W_x[i,f,g,o], b[i,f,g,o]
        self.h2h = nn.Linear(hidden_size, 4 * hidden_size, bias=False)  # W_h[i,f,g,o]
        self.reset_parameters()

    def input_projection(self, x):
        return self.x2h(x)

    def init_state(self, batch_size, device):
        zeros = torch.zeros(batch_size, self.hidden_size, device=device)
        return zeros, zeros.clone()

    def forward(self, xp_t, state):
        h_prev, c_prev = state
        gates = xp_t + self.h2h(h_prev)
        i, f, g, o = gates.chunk(4, dim=-1)
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        g = torch.tanh(g)
        c = f * c_prev + i * g
        h = o * torch.tanh(c)
        return h, (h, c)

    @torch.no_grad()
    def reset_parameters(self, scheme: str = "default", forget_bias: float = 0.0):
        H = self.hidden_size
        if scheme == "default":  # same as torch.nn.LSTM: U(-1/sqrt(H), 1/sqrt(H)) everywhere
            bound = 1.0 / math.sqrt(H)
            for p in self.parameters():
                nn.init.uniform_(p, -bound, bound)
        elif scheme in ("xavier", "orthogonal"):
            for k in range(4):  # initialise each gate block as its own (H x D) matrix
                nn.init.xavier_uniform_(self.x2h.weight[k * H:(k + 1) * H])
                if scheme == "xavier":
                    nn.init.xavier_uniform_(self.h2h.weight[k * H:(k + 1) * H])
                else:
                    nn.init.orthogonal_(self.h2h.weight[k * H:(k + 1) * H])
            nn.init.zeros_(self.x2h.bias)
        else:
            raise ValueError(f"unknown init scheme: {scheme}")
        if forget_bias:
            self.x2h.bias[H:2 * H].fill_(forget_bias)  # remember by default at the start of training


class WMCCell(VanillaLSTMCell):
    r"""Working Memory Connections, guidelines equations (7)-(12).

    Input and recurrent projections use the vanilla (i, f, g, o) layout.
    Memory projections are full H x H matrices, without extra biases:
    i/f use tanh(W_c c_prev), whereas o uses tanh(W_co c_new).
    """

    def __init__(self, input_size: int, hidden_size: int):
        # Build explicitly: the parent constructor calls reset_parameters before
        # the additional memory projections would have been created.
        RecurrentCell.__init__(self)
        self.input_size = input_size
        self.hidden_size = self.output_size = hidden_size
        self.x2h = nn.Linear(input_size, 4 * hidden_size)
        self.h2h = nn.Linear(hidden_size, 4 * hidden_size, bias=False)
        self.c2if = nn.Linear(hidden_size, 2 * hidden_size, bias=False)
        self.c2o = nn.Linear(hidden_size, hidden_size, bias=False)
        self.reset_parameters()

    def forward(self, xp_t, state):
        h_prev, c_prev = state
        i, f, g, o = (xp_t + self.h2h(h_prev)).chunk(4, dim=-1)
        ci, cf = torch.tanh(self.c2if(c_prev)).chunk(2, dim=-1)
        i = torch.sigmoid(i + ci)
        f = torch.sigmoid(f + cf)
        g = torch.tanh(g)
        c = f * c_prev + i * g
        o = torch.sigmoid(o + torch.tanh(self.c2o(c)))
        h = o * torch.tanh(c)
        return h, (h, c)

    @torch.no_grad()
    def reset_parameters(self, scheme: str = "default", forget_bias: float = 0.0):
        super().reset_parameters(scheme, forget_bias)
        # The parent's default scheme already initializes every parameter,
        # including memory weights, with the common uniform distribution.
        if scheme in ("xavier", "orthogonal"):
            init = nn.init.xavier_uniform_ if scheme == "xavier" else nn.init.orthogonal_
            for block in self.c2if.weight.chunk(2, dim=0):
                init(block)
            init(self.c2o.weight)


CELLS: dict[str, type[RecurrentCell]] = {
    "vanilla": VanillaLSTMCell,
    "wmc": WMCCell,
    # "cifg": CIFGCell, "lstmp": LSTMPCell  <- other variants
}


def make_cell(name: str, input_size: int, hidden_size: int, **kwargs) -> RecurrentCell:
    return CELLS[name](input_size, hidden_size, **kwargs)


In [ ]:
%%writefile src/model.py
"""Sequence layer and NER tagger built on top of the custom cells."""
from __future__ import annotations

import torch
from torch import nn

from cells import make_cell
from data import CASES


class RecurrentLayer(nn.Module):
    """Unrolls one cell over time (optionally a second one right-to-left).

    Padding is handled with the mask: on a padded position the state is carried over
    unchanged and the output is zero. This matters for the backward direction, which
    would otherwise start reading the sentence from the <PAD> tokens.
    """

    def __init__(self, cell: str, input_size: int, hidden_size: int, bidirectional: bool = True, **cell_kwargs):
        super().__init__()
        self.fwd = make_cell(cell, input_size, hidden_size, **cell_kwargs)
        self.bwd = make_cell(cell, input_size, hidden_size, **cell_kwargs) if bidirectional else None
        self.output_size = self.fwd.output_size * (2 if bidirectional else 1)

    @staticmethod
    def _unroll(cell, x, mask, reverse: bool):
        B, T, _ = x.shape
        xp = cell.input_projection(x)  # input terms W_x x_t + b for every t in one matmul
        state = cell.init_state(B, x.device)
        outputs = [None] * T
        for t in (range(T - 1, -1, -1) if reverse else range(T)):
            h, new_state = cell(xp[:, t], state)
            m = mask[:, t].unsqueeze(-1)
            state = tuple(torch.where(m, n, s) for n, s in zip(new_state, state))
            outputs[t] = h * m
        return torch.stack(outputs, dim=1)  # (B, T, output_size)

    def forward(self, x, mask):
        out = self._unroll(self.fwd, x, mask, reverse=False)
        if self.bwd is not None:
            out = torch.cat([out, self._unroll(self.bwd, x, mask, reverse=True)], dim=-1)
        return out

    def reset_parameters(self, scheme: str, forget_bias: float):
        for cell in (self.fwd, self.bwd):
            if cell is not None:
                cell.reset_parameters(scheme, forget_bias)


class NERTagger(nn.Module):
    """word emb ⊕ casing emb -> dropout -> [RecurrentLayer -> dropout] x L -> Linear -> tag logits."""

    def __init__(self, vocab_size: int, num_tags: int, cell: str = "vanilla", emb_dim: int = 100,
                 case_dim: int = 8, hidden_size: int = 128, num_layers: int = 1, bidirectional: bool = True,
                 dropout: float = 0.3, init: str = "default", forget_bias: float = 0.0, **cell_kwargs):
        super().__init__()
        self.word_emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.case_emb = nn.Embedding(len(CASES), case_dim, padding_idx=0) if case_dim > 0 else None
        self.dropout = nn.Dropout(dropout)
        layers, in_size = [], emb_dim + max(case_dim, 0)
        for _ in range(num_layers):
            layer = RecurrentLayer(cell, in_size, hidden_size, bidirectional, **cell_kwargs)
            layer.reset_parameters(init, forget_bias)
            layers.append(layer)
            in_size = layer.output_size
        self.layers = nn.ModuleList(layers)
        self.classifier = nn.Linear(in_size, num_tags)

    def forward(self, words, cases, mask):
        x = self.word_emb(words)
        if self.case_emb is not None:
            x = torch.cat([x, self.case_emb(cases)], dim=-1)
        x = self.dropout(x)
        for layer in self.layers:
            x = self.dropout(layer(x, mask))
        return self.classifier(x)  # (B, T, num_tags)

    def parameter_counts(self) -> dict[str, int]:
        def n(module):
            return sum(p.numel() for p in module.parameters() if p.requires_grad)

        return {"total": n(self), "recurrent": sum(n(l) for l in self.layers),
                "embeddings": n(self.word_emb) + (n(self.case_emb) if self.case_emb else 0),
                "classifier": n(self.classifier)}


In [ ]:
%%writefile src/train.py
"""Train / evaluate a recurrent NER tagger on FloraNER.

Example:
    python src/train.py --task fine --cell vanilla --hidden-size 128 --optimizer adam --lr 1e-3

Each run writes results/runs/<run_name>/{config,history,metrics}.json and appends one line
to results/summary.jsonl so that all experiments of the group can be compared side by side.
"""
from __future__ import annotations

import argparse
import json
import os
import random
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader

from data import PAD_LABEL_ID, ROOT, collate_fn, load_task
from metrics import entity_scores
from model import NERTagger

RESULTS = Path(os.environ.get("FLORANER_RESULTS", ROOT / "results"))  # e.g. a Google Drive folder on Colab


@dataclass
class Config:
    task: str = "fine"
    cell: str = "vanilla"
    # architecture
    emb_dim: int = 100
    case_dim: int = 8
    hidden_size: int = 128
    num_layers: int = 1
    bidirectional: bool = True
    dropout: float = 0.3
    init: str = "default"          # default | xavier | orthogonal
    forget_bias: float = 0.0
    lowercase: bool = True
    min_freq: int = 1
    # optimisation
    optimizer: str = "adam"        # adam | adamw | sgd | rmsprop
    lr: float = 1e-3
    weight_decay: float = 0.0
    momentum: float = 0.9
    batch_size: int = 32
    epochs: int = 60               # upper bound
    patience: int = 10             # stop after 10 epochs without val F1 gain (0 = never); best val epoch kept
    clip: float = 5.0
    scheduler: str = "none"        # none | plateau
    # bookkeeping
    seed: int = 1
    run_name: str = ""
    tensorboard: bool = False
    save_model: bool = False
    max_train_batches: int = 0     # >0: debug on a few batches only
    threads: int = 0
    tags: list = field(default_factory=list)  # free labels, e.g. ["study:optimizer"]


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_optimizer(cfg: Config, params):
    if cfg.optimizer == "adam":
        return torch.optim.Adam(params, lr=cfg.lr, weight_decay=cfg.weight_decay)
    if cfg.optimizer == "adamw":
        return torch.optim.AdamW(params, lr=cfg.lr, weight_decay=cfg.weight_decay or 1e-2)
    if cfg.optimizer == "sgd":
        return torch.optim.SGD(params, lr=cfg.lr, momentum=cfg.momentum, weight_decay=cfg.weight_decay)
    if cfg.optimizer == "rmsprop":
        return torch.optim.RMSprop(params, lr=cfg.lr, weight_decay=cfg.weight_decay)
    raise ValueError(cfg.optimizer)


@torch.no_grad()
def evaluate(model, loader, tag_vocab, device, criterion) -> dict:
    model.eval()
    y_true, y_pred, total_loss, n_tok, t0 = [], [], 0.0, 0, time.perf_counter()
    for W, C, Y, mask in loader:
        W, C, Y, mask = W.to(device), C.to(device), Y.to(device), mask.to(device)
        logits = model(W, C, mask)
        total_loss += criterion(logits.reshape(-1, logits.size(-1)), Y.reshape(-1)).item() * mask.sum().item()
        n_tok += mask.sum().item()
        pred = logits.argmax(-1)
        for p, y, m in zip(pred.tolist(), Y.tolist(), mask.tolist()):
            L = sum(m)
            y_true.append([tag_vocab.itos[i] for i in y[:L]])
            y_pred.append([tag_vocab.itos[i] for i in p[:L]])
    elapsed = time.perf_counter() - t0
    flat_t = [t for s in y_true for t in s]
    flat_p = [t for s in y_pred for t in s]
    scores = entity_scores(y_true, y_pred)
    return {
        "loss": total_loss / n_tok,
        "precision": scores["precision"],
        "recall": scores["recall"],
        "f1": scores["f1"],
        "macro_f1": scores["macro_f1"],
        "token_accuracy": sum(a == b for a, b in zip(flat_t, flat_p)) / len(flat_t),
        "per_type": {k: {m: round(float(v), 4) for m, v in d.items()} for k, d in scores["per_type"].items()},
        "tokens_per_sec": n_tok / elapsed,
    }


def run(cfg: Config) -> dict:
    set_seed(cfg.seed)
    if cfg.threads:
        torch.set_num_threads(cfg.threads)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cfg.run_name = cfg.run_name or (f"{cfg.task}_{cfg.cell}_h{cfg.hidden_size}_l{cfg.num_layers}"
                                    f"{'_bi' if cfg.bidirectional else ''}_{cfg.optimizer}_lr{cfg.lr}"
                                    f"_bs{cfg.batch_size}_{cfg.init}_s{cfg.seed}")
    out_dir = RESULTS / "runs" / cfg.run_name
    out_dir.mkdir(parents=True, exist_ok=True)

    data, word_vocab, tag_vocab = load_task(cfg.task, cfg.lowercase, cfg.min_freq)
    g = torch.Generator().manual_seed(cfg.seed)
    train_loader = DataLoader(data["train"], cfg.batch_size, shuffle=True, collate_fn=collate_fn, generator=g)
    val_loader = DataLoader(data["val"], 64, collate_fn=collate_fn)
    test_loader = DataLoader(data["test"], 64, collate_fn=collate_fn)

    model = NERTagger(len(word_vocab), len(tag_vocab), cell=cfg.cell, emb_dim=cfg.emb_dim, case_dim=cfg.case_dim,
                      hidden_size=cfg.hidden_size, num_layers=cfg.num_layers, bidirectional=cfg.bidirectional,
                      dropout=cfg.dropout, init=cfg.init, forget_bias=cfg.forget_bias).to(device)
    n_params = model.parameter_counts()
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_LABEL_ID)
    optimizer = make_optimizer(cfg, model.parameters())
    scheduler = (torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
                 if cfg.scheduler == "plateau" else None)
    writer = None
    if cfg.tensorboard:
        from torch.utils.tensorboard import SummaryWriter
        writer = SummaryWriter(RESULTS / "tensorboard" / cfg.run_name)

    print(f"[{cfg.run_name}] device={device} params={n_params} train_segments={len(data['train'])}")
    history, best_f1, best_state, best_epoch, bad_epochs = [], -1.0, None, 0, 0
    for epoch in range(1, cfg.epochs + 1):
        model.train()
        t0, train_loss, n_tok = time.perf_counter(), 0.0, 0
        for b, (W, C, Y, mask) in enumerate(train_loader):
            if cfg.max_train_batches and b >= cfg.max_train_batches:
                break
            W, C, Y, mask = W.to(device), C.to(device), Y.to(device), mask.to(device)
            logits = model(W, C, mask)
            loss = criterion(logits.reshape(-1, logits.size(-1)), Y.reshape(-1))
            optimizer.zero_grad()
            loss.backward()
            if cfg.clip:
                nn.utils.clip_grad_norm_(model.parameters(), cfg.clip)
            optimizer.step()
            train_loss += loss.item() * mask.sum().item()
            n_tok += mask.sum().item()
        epoch_time = time.perf_counter() - t0
        val = evaluate(model, val_loader, tag_vocab, device, criterion)
        row = {"epoch": epoch, "train_loss": train_loss / n_tok, "val_loss": val["loss"], "val_f1": val["f1"],
               "val_macro_f1": val["macro_f1"], "val_precision": val["precision"], "val_recall": val["recall"],
               "lr": optimizer.param_groups[0]["lr"], "epoch_time_s": epoch_time}
        history.append(row)
        print(f"  epoch {epoch:2d} | train_loss {row['train_loss']:.4f} | val_loss {val['loss']:.4f} "
              f"| val F1 {val['f1']:.4f} (macro {val['macro_f1']:.4f}) | {epoch_time:.1f}s")
        if writer:
            for k, v in row.items():
                if k != "epoch":
                    writer.add_scalar(k, v, epoch)
        if scheduler:
            scheduler.step(val["f1"])
        if val["f1"] > best_f1:
            best_f1, best_epoch, bad_epochs = val["f1"], epoch, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1
            if cfg.patience and bad_epochs >= cfg.patience:
                print(f"  early stopping (best epoch {best_epoch})")
                break

    model.load_state_dict(best_state)
    val = evaluate(model, val_loader, tag_vocab, device, criterion)
    test = evaluate(model, test_loader, tag_vocab, device, criterion)
    print(f"  BEST epoch {best_epoch} | val F1 {val['f1']:.4f} | TEST F1 {test['f1']:.4f} "
          f"(P {test['precision']:.4f} R {test['recall']:.4f} macro {test['macro_f1']:.4f})")

    epoch_times = [h["epoch_time_s"] for h in history]
    metrics = {"best_epoch": best_epoch, "epochs_run": len(history), "params": n_params,
               "mean_epoch_time_s": sum(epoch_times) / len(epoch_times), "device": str(device),
               "val": val, "test": test}
    for name, obj in (("config", asdict(cfg)), ("history", history), ("metrics", metrics)):
        with open(out_dir / f"{name}.json", "w") as f:
            json.dump(obj, f, indent=2, ensure_ascii=False)
    if cfg.save_model:
        torch.save({"state_dict": best_state, "config": asdict(cfg), "word_vocab": word_vocab.itos,
                    "tag_vocab": tag_vocab.itos}, out_dir / "model.pt")
    summary = {**asdict(cfg), "params_total": n_params["total"], "params_recurrent": n_params["recurrent"],
               "best_epoch": best_epoch, "mean_epoch_time_s": metrics["mean_epoch_time_s"], "device": str(device),
               **{f"val_{k}": val[k] for k in ("precision", "recall", "f1", "macro_f1")},
               **{f"test_{k}": test[k] for k in ("precision", "recall", "f1", "macro_f1", "token_accuracy",
                                                  "tokens_per_sec")}}
    with open(RESULTS / "summary.jsonl", "a") as f:
        f.write(json.dumps(summary, ensure_ascii=False) + "\n")
    if writer:
        writer.close()
    return summary


def parse_args() -> Config:
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    for f in Config.__dataclass_fields__.values():
        flag = "--" + f.name.replace("_", "-")
        if f.type in ("bool", bool):
            p.add_argument(flag, action=argparse.BooleanOptionalAction, default=f.default)
        elif f.name == "tags":
            p.add_argument(flag, nargs="*", default=[])
        else:
            typ = {"int": int, "float": float, "str": str}[f.type if isinstance(f.type, str) else f.type.__name__]
            p.add_argument(flag, type=typ, default=f.default)
    return Config(**vars(p.parse_args()))


if __name__ == "__main__":
    run(parse_args())


In [ ]:
%%writefile src/sweep.py
"""One-factor-at-a-time hyperparameter studies around the baseline, several seeds each.

Usage:
    python src/sweep.py --group 1                     # the share of Colab account 1 (see GROUPS)
    python src/sweep.py --study optimizer lr          # some studies
    python src/sweep.py --study all --seeds 1 2 3     # everything
    python src/sweep.py --group 2 --list              # print the plan
    python src/sweep.py --cell cifg --study baseline  # a new cell under the common protocol (3 runs)

Every run trains for at most 60 epochs and stops after 10 epochs without validation-F1
improvement; the best validation epoch is kept. Runs are named deterministically from their
settings; a run already present in results/summary.jsonl is skipped, so an interrupted sweep
simply resumes. Hyperparameters are chosen on the validation F1; the test split is only reported.
"""
from __future__ import annotations

import argparse
import json

from train import RESULTS, Config, run

BASELINE = dict(cell="vanilla", emb_dim=100, case_dim=8, hidden_size=128, num_layers=1, bidirectional=True,
                dropout=0.3, init="default", forget_bias=0.0, optimizer="adam", lr=1e-3, batch_size=32)

STUDIES: dict[str, list[dict]] = {
    "baseline": [dict()],  # the comparison run of every cell: common settings x seeds 1, 2, 3
    "optimizer": [dict(optimizer="adam", lr=1e-3), dict(optimizer="adamw", lr=1e-3),
                  dict(optimizer="rmsprop", lr=1e-3), dict(optimizer="sgd", lr=0.1)],
    "lr": [dict(lr=lr) for lr in (1e-2, 3e-3, 1e-3, 3e-4)],
    "batch_size": [dict(batch_size=b) for b in (16, 32, 64, 128)],
    "init": [dict(init="default"), dict(init="xavier"), dict(init="orthogonal"),
             dict(init="orthogonal", forget_bias=1.0)],
    "hidden_size": [dict(hidden_size=h) for h in (64, 128, 256)],
    "num_layers": [dict(num_layers=n) for n in (1, 2)],
    "direction": [dict(bidirectional=False), dict(bidirectional=True)],
    "dropout": [dict(dropout=d) for d in (0.0, 0.3, 0.5)],
    "case_features": [dict(case_dim=0), dict(case_dim=8)],
}

# Share of the sweep per Colab account (~20 runs each). The baseline is trained by account 1 only;
# the other accounts skip it and their results are merged afterwards (src/merge_results.py).
GROUPS: dict[str, tuple[list[str], bool]] = {  # name -> (studies, exclude baseline)
    "1": (["optimizer", "lr"], False),
    "2": (["batch_size", "init"], True),
    "3": (["hidden_size", "num_layers", "direction", "dropout", "case_features"], True),
    "all": (list(STUDIES), False),
}

SHORT = {"optimizer": "opt", "lr": "lr", "batch_size": "bs", "init": "init", "forget_bias": "fb",
         "hidden_size": "h", "num_layers": "l", "bidirectional": "bi", "dropout": "do", "case_dim": "case",
         "emb_dim": "emb", "cell": "cell"}


def is_baseline(override: dict) -> bool:
    return all(BASELINE.get(k) == v for k, v in override.items())


def run_name(task: str, override: dict, seed: int, cell: str = "vanilla") -> str:
    diff = {k: v for k, v in override.items() if BASELINE.get(k) != v}
    suffix = "_".join(f"{SHORT.get(k, k)}{v}" for k, v in sorted(diff.items())) or "baseline"
    return f"{task}_{cell}_{suffix}_s{seed}"


def make_config(task: str, override: dict, seed: int, cell: str, epochs: int, patience: int) -> Config:
    params = {**BASELINE, "cell": cell, **override}
    return Config(task=task, seed=seed, epochs=epochs, patience=patience,
                  run_name=run_name(task, override, seed, cell), **params)


def done_runs() -> set[str]:
    path = RESULTS / "summary.jsonl"
    if not path.exists():
        return set()
    return {json.loads(line)["run_name"] for line in open(path)}


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--group", choices=list(GROUPS), help="predefined share of the sweep (overrides --study)")
    p.add_argument("--study", nargs="+", default=["all"])
    p.add_argument("--exclude-baseline", action="store_true", help="skip runs equal to the baseline")
    p.add_argument("--task", default="fine")
    p.add_argument("--cell", default="vanilla")
    p.add_argument("--seeds", nargs="+", type=int, default=[1, 2, 3])
    p.add_argument("--epochs", type=int, default=60)
    p.add_argument("--patience", type=int, default=10, help="0 = always train all epochs")
    p.add_argument("--list", action="store_true")
    args = p.parse_args()

    if args.group:
        studies, exclude_baseline = GROUPS[args.group]
    else:
        studies = list(STUDIES) if args.study == ["all"] else args.study
        exclude_baseline = args.exclude_baseline
    plan, seen = [], set()
    for study in studies:
        for override in STUDIES[study]:
            if exclude_baseline and is_baseline(override):
                continue
            for seed in args.seeds:
                cfg = make_config(args.task, override, seed, args.cell, args.epochs, args.patience)
                if cfg.run_name not in seen:
                    seen.add(cfg.run_name)
                    plan.append(cfg)
    done = done_runs()
    todo = [c for c in plan if c.run_name not in done]
    print(f"studies: {', '.join(studies)} | {len(plan)} runs planned, {len(plan) - len(todo)} already done, "
          f"{len(todo)} to go")
    if args.list:
        for c in plan:
            print(("  [done] " if c.run_name in done else "  [todo] ") + c.run_name)
        return
    for k, cfg in enumerate(todo, 1):
        print(f"\n===== run {k}/{len(todo)}: {cfg.run_name}")
        run(cfg)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile tests/test_model.py
"""Sanity checks for the from-scratch LSTM. Run: python -m pytest tests -q

nn.LSTMCell is used here ONLY as a numerical reference, never inside the model.
"""
import sys
from pathlib import Path

import pytest
import torch
from torch import nn

sys.path.insert(0, str(Path(__file__).resolve().parent.parent / "src"))
from cells import VanillaLSTMCell  # noqa: E402
from data import collate_fn, spans_to_bio  # noqa: E402
from model import NERTagger, RecurrentLayer  # noqa: E402


def test_vanilla_cell_matches_pytorch_lstmcell():
    torch.manual_seed(0)
    D, H, B, T = 7, 5, 3, 12
    ours, ref = VanillaLSTMCell(D, H), nn.LSTMCell(D, H)
    with torch.no_grad():  # same gate order (i, f, g, o) -> copy weights directly
        ref.weight_ih.copy_(ours.x2h.weight)
        ref.bias_ih.copy_(ours.x2h.bias)
        ref.weight_hh.copy_(ours.h2h.weight)
        ref.bias_hh.zero_()
    x = torch.randn(B, T, D)
    xp = ours.input_projection(x)
    state, (h_ref, c_ref) = ours.init_state(B, "cpu"), (torch.zeros(B, H), torch.zeros(B, H))
    for t in range(T):
        h, state = ours(xp[:, t], state)
        h_ref, c_ref = ref(x[:, t], (h_ref, c_ref))
        assert torch.allclose(h, h_ref, atol=1e-6) and torch.allclose(state[1], c_ref, atol=1e-6)


@pytest.mark.parametrize("bidirectional", [False, True])
def test_padding_does_not_change_outputs(bidirectional):
    torch.manual_seed(0)
    layer = RecurrentLayer("vanilla", 4, 6, bidirectional=bidirectional)
    short = torch.randn(1, 5, 4)
    alone = layer(short, torch.ones(1, 5, dtype=torch.bool))
    batch = torch.cat([torch.cat([short, torch.randn(1, 4, 4)], 1), torch.randn(1, 9, 4)])
    mask = torch.tensor([[True] * 5 + [False] * 4, [True] * 9])
    padded = layer(batch, mask)
    assert torch.allclose(alone[0], padded[0, :5], atol=1e-6)
    assert torch.all(padded[0, 5:] == 0)


def test_parameter_count_matches_formula():
    D, H = 108, 128
    cell = VanillaLSTMCell(D, H)
    assert cell.recurrent_parameter_count() == 4 * (D * H + H * H + H)


def test_forward_shapes_and_loss_ignores_padding():
    torch.manual_seed(0)
    model = NERTagger(vocab_size=50, num_tags=5, hidden_size=8, emb_dim=6, case_dim=3, num_layers=2)
    batch = [(torch.tensor([3, 4, 5]), torch.tensor([1, 1, 2]), torch.tensor([0, 1, 2])),
             (torch.tensor([6]), torch.tensor([1]), torch.tensor([3]))]
    W, C, Y, mask = collate_fn(batch)
    logits = model(W, C, mask)
    assert logits.shape == (2, 3, 5) and Y[1, 1:].eq(-100).all()
    loss = nn.CrossEntropyLoss(ignore_index=-100)(logits.reshape(-1, 5), Y.reshape(-1))
    loss.backward()
    assert torch.isfinite(loss)


def test_entity_scores_match_seqeval():
    seqeval = pytest.importorskip("seqeval.metrics")  # reference only; not needed to train
    import random

    from metrics import entity_scores, get_entities

    assert get_entities(["B-A", "I-A", "O", "I-B", "I-A", "B-A"]) == [("A", 0, 1), ("B", 3, 3), ("A", 4, 4),
                                                                       ("A", 5, 5)]
    rng = random.Random(0)
    tags = ["O", "O", "O", "B-A", "I-A", "B-B", "I-B", "B-C"]
    y_true = [[rng.choice(tags) for _ in range(rng.randint(1, 15))] for _ in range(2000)]
    y_pred = [[t if rng.random() < 0.7 else rng.choice(tags) for t in s] for s in y_true]
    ours = entity_scores(y_true, y_pred)
    assert ours["precision"] == pytest.approx(seqeval.precision_score(y_true, y_pred))
    assert ours["recall"] == pytest.approx(seqeval.recall_score(y_true, y_pred))
    assert ours["f1"] == pytest.approx(seqeval.f1_score(y_true, y_pred))
    assert ours["macro_f1"] == pytest.approx(seqeval.f1_score(y_true, y_pred, average="macro"))
    report = seqeval.classification_report(y_true, y_pred, output_dict=True)
    for t, d in ours["per_type"].items():
        assert d["f1-score"] == pytest.approx(report[t]["f1-score"]) and d["support"] == report[t]["support"]


def test_spans_to_bio():
    text = "feuilles obovées-elliptiques, 20 cm de longueur"
    spans = [(0, 8, "ORGANE"), (9, 28, "FORME"), (30, 47, "MESURE")]
    tokens, tags, _ = spans_to_bio(text, spans)
    assert tokens == ["feuilles", "obovées", "-", "elliptiques", ",", "20", "cm", "de", "longueur"]
    assert tags == ["B-ORGANE", "B-FORME", "I-FORME", "I-FORME", "O",
                    "B-MESURE", "I-MESURE", "I-MESURE", "I-MESURE"]


In [ ]:
%%writefile tests/test_wmc.py
"""Independent equation and integration checks for the duo's WMC cell."""
import sys
from pathlib import Path

import pytest
import torch
from torch.nn import functional as F

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))
from cells import WMCCell, VanillaLSTMCell
from model import RecurrentLayer, NERTagger


def reference(cell, x, h, c):
    H = cell.hidden_size
    # Separate affine operations directly mirror equations 7-12.
    def affine(k):
        sl = slice(k * H, (k + 1) * H)
        return F.linear(x, cell.x2h.weight[sl], cell.x2h.bias[sl]) + F.linear(h, cell.h2h.weight[sl])
    i = torch.sigmoid(affine(0) + torch.tanh(F.linear(c, cell.c2if.weight[:H])))
    f = torch.sigmoid(affine(1) + torch.tanh(F.linear(c, cell.c2if.weight[H:])))
    g = torch.tanh(affine(2))
    new_c = f * c + i * g
    o = torch.sigmoid(affine(3) + torch.tanh(F.linear(new_c, cell.c2o.weight)))
    return o * torch.tanh(new_c), new_c


def test_equations_and_gradients():
    torch.manual_seed(13)
    cell = WMCCell(3, 4).double()
    x = torch.randn(2, 3, dtype=torch.double, requires_grad=True)
    h = torch.randn(2, 4, dtype=torch.double, requires_grad=True)
    c = torch.randn(2, 4, dtype=torch.double, requires_grad=True)
    actual, state = cell(cell.input_projection(x), (h, c))
    expected, expected_c = reference(cell, x, h, c)
    torch.testing.assert_close(actual, expected)
    torch.testing.assert_close(state[1], expected_c)
    inputs = (x, h, c, *cell.parameters())
    ga = torch.autograd.grad(actual.square().sum() + state[1].square().sum(), inputs, retain_graph=True)
    ge = torch.autograd.grad(expected.square().sum() + expected_c.square().sum(), inputs)
    for a, e in zip(ga, ge):
        torch.testing.assert_close(a, e)
        assert torch.isfinite(a).all()


def test_output_gate_uses_updated_memory():
    cell = WMCCell(1, 1)
    with torch.no_grad():
        for p in cell.parameters():
            p.zero_()
        cell.x2h.bias[2] = 1.0
        cell.c2o.weight.fill_(2.0)
    h, (_, c) = cell(cell.input_projection(torch.zeros(1, 1)), cell.init_state(1, 'cpu'))
    expected = torch.sigmoid(torch.tanh(2 * c)) * torch.tanh(c)
    torch.testing.assert_close(h, expected)
    assert not torch.allclose(h, 0.5 * torch.tanh(c))


def test_zero_memory_connections_reduce_to_vanilla():
    torch.manual_seed(9)
    wmc, vanilla = WMCCell(3, 5), VanillaLSTMCell(3, 5)
    with torch.no_grad():
        vanilla.x2h.load_state_dict(wmc.x2h.state_dict())
        vanilla.h2h.load_state_dict(wmc.h2h.state_dict())
        wmc.c2if.weight.zero_()
        wmc.c2o.weight.zero_()
    x = torch.randn(2, 7, 3)
    sw, sv = wmc.init_state(2, 'cpu'), vanilla.init_state(2, 'cpu')
    for t in range(7):
        hw, sw = wmc(wmc.input_projection(x[:, t]), sw)
        hv, sv = vanilla(vanilla.input_projection(x[:, t]), sv)
        torch.testing.assert_close(hw, hv)
        torch.testing.assert_close(sw[1], sv[1])


@pytest.mark.parametrize('scheme', ['default', 'xavier', 'orthogonal'])
def test_initialization_and_parameter_count(scheme):
    cell = WMCCell(108, 128)
    cell.reset_parameters(scheme, forget_bias=1.0)
    assert cell.recurrent_parameter_count() == 4 * (108 * 128 + 128 * 128 + 128) + 3 * 128**2
    torch.testing.assert_close(cell.x2h.bias[128:256], torch.ones(128))
    assert all(torch.isfinite(p).all() for p in cell.parameters())


@pytest.mark.parametrize('bidirectional', [False, True])
def test_padding_invariance(bidirectional):
    torch.manual_seed(8)
    layer = RecurrentLayer('wmc', 3, 5, bidirectional)
    short = torch.randn(1, 4, 3)
    alone = layer(short, torch.ones(1, 4, dtype=torch.bool))
    batch = torch.cat([F.pad(short, (0, 0, 0, 3)), torch.randn(1, 7, 3)])
    mask = torch.tensor([[True]*4 + [False]*3, [True]*7])
    padded = layer(batch, mask)
    torch.testing.assert_close(alone[0], padded[0, :4])
    assert padded[0, 4:].eq(0).all()


def test_tagger_backward_and_counts():
    model = NERTagger(5524, 19, cell='wmc')
    counts = model.parameter_counts()
    assert counts['recurrent'] == 340992
    assert counts['total'] == sum(p.numel() for p in model.parameters())
    small = NERTagger(20, 5, cell='wmc', emb_dim=4, hidden_size=6, num_layers=2)
    logits = small(torch.tensor([[1, 2, 3], [4, 0, 0]]), torch.ones(2, 3, dtype=torch.long),
                   torch.tensor([[True, True, True], [True, False, False]]))
    loss = F.cross_entropy(logits.reshape(-1, 5), torch.tensor([0, 1, 2, 3, -100, -100]))
    loss.backward()
    assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in small.parameters())


In [ ]:
%%writefile tests/test_wmc_summary.py
"""Check handoff summary rejects partial seeds and protocol drift."""
import json
import sys
from pathlib import Path
import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))
import summarize_wmc
from train import Config
from dataclasses import asdict


def write_run(root, seed):
    folder = root / 'runs' / f'fine_wmc_baseline_s{seed}'
    folder.mkdir(parents=True)
    cfg = asdict(Config(cell='wmc', seed=seed))
    (folder/'config.json').write_text(json.dumps(cfg))
    metrics = {'test':dict(precision=.9, recall=.8, f1=.84+seed*.01, macro_f1=.7),
               'best_epoch':20,'mean_epoch_time_s':15,'params':{'total':898339,'recurrent':340992}}
    (folder/'metrics.json').write_text(json.dumps(metrics))
    return folder


def test_summary_requires_all_three_seeds(tmp_path, monkeypatch):
    monkeypatch.setattr(summarize_wmc, 'RESULTS', tmp_path)
    write_run(tmp_path,1)
    with pytest.raises(FileNotFoundError):
        summarize_wmc.main()
    assert not (tmp_path/'wmc_summary.md').exists()


def test_summary_rejects_protocol_drift(tmp_path, monkeypatch):
    monkeypatch.setattr(summarize_wmc, 'RESULTS', tmp_path)
    for seed in (1,2,3):
        folder = write_run(tmp_path,seed)
    cfg=json.loads((folder/'config.json').read_text())
    cfg['max_train_batches']=2
    (folder/'config.json').write_text(json.dumps(cfg))
    with pytest.raises(ValueError,match='max_train_batches'):
        summarize_wmc.main()
    assert not (tmp_path/'wmc_summary.md').exists()


def test_summary_aggregates_complete_runs(tmp_path, monkeypatch):
    monkeypatch.setattr(summarize_wmc, 'RESULTS', tmp_path)
    for seed in (1,2,3):
        write_run(tmp_path,seed)
    summarize_wmc.main()
    text=(tmp_path/'wmc_summary.md').read_text(encoding='utf-8')
    assert '86.00 ± 1.00%' in text
    assert '898339' in text


In [ ]:
%%writefile src/summarize_wmc.py
"""Summarize only the WMC baseline; reject incomplete or non-common runs."""
import json
import statistics
from train import RESULTS
from sweep import BASELINE


def main():
    rows = []
    for seed in (1, 2, 3):
        folder = RESULTS / 'runs' / f'fine_wmc_baseline_s{seed}'
        cfg = json.loads((folder / 'config.json').read_text())
        expected = {**BASELINE, 'cell': 'wmc', 'task': 'fine', 'seed': seed, 'epochs': 60, 'patience': 10,
                    'max_train_batches': 0, 'scheduler': 'none', 'weight_decay': 0.0,
                    'clip': 5.0, 'lowercase': True, 'min_freq': 1}
        for key, value in expected.items():
            if cfg[key] != value:
                raise ValueError(f'{folder.name}: {key}={cfg[key]!r}; expected {value!r}')
        metrics = json.loads((folder / 'metrics.json').read_text())
        rows.append(metrics)
    lines = ['# WMC baseline — Said & MS', '',
             'Common protocol; seeds 1, 2, 3. Test metrics use the best validation epoch.', '',
             '| Seed | Test P (%) | Test R (%) | Test F1 (%) | Macro F1 (%) | Best epoch | s/epoch |',
             '|---|---|---|---|---|---|---|']
    for seed, row in zip((1, 2, 3), rows):
        t = row['test']
        lines.append(f"| {seed} | {100*t['precision']:.2f} | {100*t['recall']:.2f} | {100*t['f1']:.2f} | {100*t['macro_f1']:.2f} | {row['best_epoch']} | {row['mean_epoch_time_s']:.2f} |")
    lines += ['', 'Mean ± sample standard deviation across three seeds:']
    for key in ('precision', 'recall', 'f1', 'macro_f1'):
        vals = [100*r['test'][key] for r in rows]
        lines.append(f'- Test {key}: {statistics.mean(vals):.2f} ± {statistics.stdev(vals):.2f}%')
    if any(r['params'] != rows[0]['params'] for r in rows):
        raise ValueError('Parameter counts differ across seeds')
    lines += ['', 'Trainable parameters: ' + json.dumps(rows[0]['params']), '',
              'Collective comparison and interpretation remain group work.']
    out = RESULTS / 'wmc_summary.md'
    out.write_text('\n'.join(lines), encoding='utf-8')
    print('\n'.join(lines))

if __name__ == '__main__':
    main()


## Verify and prepare the common data

In [ ]:
run('-m', 'pytest', 'tests', '-q')
run('src/download_data.py')

## Three full WMC runs
Completed runs are skipped on rerun. An interrupted run restarts from its seed; partial epochs are not checkpointed.

In [ ]:
run('-u', 'src/sweep.py', '--cell', 'wmc', '--study', 'baseline', '--seeds', '1', '2', '3', '--epochs', '60', '--patience', '10')

## WMC results and handoff archive

In [ ]:
run('src/summarize_wmc.py')
import shutil
from google.colab import files
archive = shutil.make_archive('/content/floraner_wmc_results', 'zip', os.environ['FLORANER_RESULTS'])
files.download(archive)
